# TIFF → nucleus_boundaries.parquet
Extracts cell polygon boundaries and centroids from CellPose TIFF segmentation masks.

**Inputs:** `.tif` masks (one per sample, named `{sample_id}_DAPI*.tif`)  
**Outputs:** `{INPUT_DIR}/{sample_id}/nucleus_boundaries.parquet` — columns: `geometry`, `EntityID`, `centroid_x`, `centroid_y`  
**Config:** fill in `MC_SEGMENTATIONS_DIR` and `INPUT_DIR` in the Configuration cell below

## Function definition
`tiff_to_parquet` — reads a single TIFF mask, extracts polygons per cell via `rasterio.shapes`, keeps the largest polygon when a nucleus is fragmented, and writes a GeoParquet.

In [ ]:
def tiff_to_parquet(file_name, input_folder, output_folder):
    import os
    import tifffile as tiff
    import numpy as np
    import geopandas as gpd
    from shapely.geometry import shape
    from rasterio.features import shapes
    import pandas as pd
    from collections import defaultdict
    from tqdm import tqdm
    from pathlib import Path

    tiff_path = os.path.join(input_folder, file_name)
    # Sample ID is everything before '_DAPI' in the filename
    output_name = file_name.split('_DAPI')[0]
    output_dir = Path(output_folder) / output_name
    output_dir.mkdir(parents=True, exist_ok=True)
    parquet_path = output_dir / "nucleus_boundaries.parquet"

    print("Loading TIFF...")
    mask = tiff.imread(tiff_path)
    print('tiff loaded successfully')

    assert mask.ndim == 2, "Only 2D masks supported"
    assert mask.dtype.kind in ("i", "u"), "Mask must be integer-labeled"

    print("Extracting polygons...")

    # rasterio.shapes yields (geometry, pixel_value) for each connected region of identical values
    geom_parts = defaultdict(list)
    num_shapes = len(np.unique(mask)) - 1

    for geom, value in tqdm(shapes(mask, mask=mask > 0), total=num_shapes, desc="Extracting shapes"):
        entity_id = int(value)
        poly = shape(geom)
        if poly.is_empty:
            continue
        if poly.geom_type == "Polygon":
            geom_parts[entity_id].append(poly)
        elif poly.geom_type == "MultiPolygon":
            geom_parts[entity_id].extend(list(poly.geoms))

    print(f"Found {len(geom_parts)} unique nuclei")
    records = []

    for entity_id, polygons in tqdm(geom_parts.items(), desc="Iterating through nuclei"):
        for poly in polygons:
            centroid = poly.centroid
            records.append({
                "EntityID": entity_id,
                "geometry": poly,
                "centroid_x": centroid.x,
                "centroid_y": centroid.y
            })

    gdf = gpd.GeoDataFrame(records, geometry="geometry", crs=None)
    gdf = gdf[["geometry", "EntityID", "centroid_x", "centroid_y"]]

    # Keep only the largest polygon per cell (handles fragmented nuclear outlines)
    gdf["area"] = gdf.geometry.area
    gdf_clean = gdf.loc[gdf.groupby("EntityID")["area"].idxmax()].copy()
    gdf_clean = gdf_clean.drop(columns="area")

    print(f"Writing Parquet... {parquet_path}")
    gdf_clean.to_parquet(parquet_path, index=False)
    print("Done.")


Main script for converting tif to parquet

In [ ]:
import os
from pathlib import Path

# --- CONFIGURATION: fill in paths before running ---
MC_SEGMENTATIONS_DIR = Path("")  # directory containing .tif segmentation masks
INPUT_DIR = Path("")              # output directory (input_for_segger)

# Get all .tif files
tif_files = [f for f in os.listdir(MC_SEGMENTATIONS_DIR) if f.endswith(".tif")]

# Loop through and convert each
for file_name in tif_files:
    tiff_to_parquet(file_name, str(MC_SEGMENTATIONS_DIR), str(INPUT_DIR))
